# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [11]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score

In [12]:


# 1. Cargamos el set de entrenamiento
train_set = pd.read_csv("../data/interim/train_set.csv")

# 2. Separamos las pistas (X) del objetivo a adivinar (y)
# A X le quitamos el precio. A y le dejamos SOLO el precio.
X_train = train_set.drop("median_house_value", axis=1)
y_train = train_set["median_house_value"].copy()

print(f"Tamaño de X_train (Pistas): {X_train.shape}")
print(f"Tamaño de y_train (Respuestas): {y_train.shape}")

Tamaño de X_train (Pistas): (16512, 9)
Tamaño de y_train (Respuestas): (16512,)


In [13]:
# A. Definimos la clase que crea las columnas nuevas (las matemáticas)
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CreadorAtributos(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self  
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
        return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]

# B. Separamos qué columnas son números y cuáles son texto
housing_num = X_train.select_dtypes(include=[np.number])
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

# C. Armamos la máquina para los números (Imputar nulos -> Crear Columnas -> Escalar)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CreadorAtributos()),
    ('std_scaler', StandardScaler()),
])

# D. Armamos la Máquina Maestra (Junta los números limpios con el texto codificado)
pipeline_maestro = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

# E. ¡PASAMOS LOS DATOS POR LA MÁQUINA!
X_train_preparado = pipeline_maestro.fit_transform(X_train)

print("¡Datos limpios y listos!")
print(f"Forma final: {X_train_preparado.shape}")

¡Datos limpios y listos!
Forma final: (16512, 16)


In [14]:
# 1. Creamos los modelos
lin_reg = LinearRegression()
tree_reg = DecisionTreeRegressor(random_state=42)
forest_reg = RandomForestRegressor(random_state=42)

modelos = {
    "Regresión Lineal": lin_reg,
    "Árbol de Decisión": tree_reg,
    "Random Forest": forest_reg
}

# 2. Los evaluamos con Validación Cruzada (Cross Validation)
print("Evaluando modelos")
for nombre, modelo in modelos.items():
    # Usamos el X_train_preparado!
    scores = cross_val_score(modelo, X_train_preparado, y_train, 
                             scoring="neg_mean_squared_error", cv=5)
    rmse_scores = np.sqrt(-scores)
    print(f"{nombre} -> RMSE Promedio: ${rmse_scores.mean():,.2f}")

Evaluando modelos... (Puede tardar un minuto)
Regresión Lineal -> RMSE Promedio: $67,944.32
Árbol de Decisión -> RMSE Promedio: $71,339.05
Random Forest -> RMSE Promedio: $50,070.94


In [15]:
# 1. Entrenamos al ganador oficial con todos los datos de entrenamiento
modelo_final = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_final.fit(X_train_preparado, y_train)

# 2. Traemos el archivo de prueba (El que el modelo JAMÁS ha visto)
test_set = pd.read_csv("../data/interim/test_set.csv")

# 3. Lo separamos en Pistas (X) y Respuesta Real (y)
X_test = test_set.drop("median_house_value", axis=1)
y_test = test_set["median_house_value"].copy()

# 4. ¡EL TRUCO DE MAGIA! 
# Pasamos las pistas del test por nuestra Máquina Maestra. 
# Usamos .transform() (NO fit_transform) para que use las reglas que ya aprendió.
X_test_preparado = pipeline_maestro.transform(X_test)

# 5. Predecimos y medimos el error final
predicciones = modelo_final.predict(X_test_preparado)
rmse_final = np.sqrt(mean_squared_error(y_test, predicciones))

print("=========================================")
print(f"🏆 Error RMSE Final en Producción: ${rmse_final:,.2f}")
print("=========================================")

🏆 Error RMSE Final en Producción: $50,344.55


### Benchmark y Conclusión Final

### 5. Conclusión y Benchmark de Modelos

**Análisis de Validación Cruzada (Modelos Base):**
Durante la fase de experimentación, se evaluaron tres algoritmos utilizando Validación Cruzada (Cross Validation) para garantizar una métrica robusta y evitar sesgos en la medición del Error Cuadrático Medio (RMSE):

* **Regresión Lineal:** Presentó un RMSE promedio de **$67,944.32**. Este alto margen de error constante indica un claro caso de **subajuste (underfitting)**. La relación entre las características de los distritos y el precio de las viviendas no es puramente lineal, por lo que el modelo resulta demasiado simple para capturar la complejidad de los datos.
* **Árbol de Decisión (Decision Tree):** Arrojó el peor desempeño en validación con un RMSE promedio de **$71,339.05**. Aunque los árboles de decisión suelen memorizar los datos de entrenamiento, al validarlos con pliegues de datos cruzados demuestran un severo **sobreajuste (overfitting)**, perdiendo toda capacidad de generalizar frente a datos nuevos.
* **Random Forest:** Se posicionó como el modelo ganador indiscutible con un RMSE promedio de **$50,070.94**. Al ser un método de ensamble, logra promediar las predicciones de múltiples árboles, mitigando el sobreajuste del Árbol de Decisión individual y capturando relaciones complejas de manera mucho más efectiva.

**Desempeño Final en Producción (Set de Prueba):**
Se seleccionó el **Random Forest Regressor** como nuestro algoritmo definitivo. Tras procesar los datos a través del pipeline de transformaciones y evaluar el modelo contra el Set de Prueba (datos reservados que el algoritmo nunca había visto), se obtuvo un **Error RMSE Final de $50,344.55**.

**Conclusión de Negocio:**
El error final en el set de prueba (~$50,344) se mantiene altamente consistente con el desempeño promedio de la validación cruzada (~$50,070). Esta cercanía entre ambas métricas confirma que **nuestro modelo generaliza correctamente y está libre de sobreajuste o subajuste severo**. En el contexto del mercado inmobiliario de California, este margen de error proporciona una línea base algorítmica muy sólida para estimar precios automatizados, permitiendo avanzar con confianza hacia la fase de despliegue del modelo en una API para su consumo en aplicaciones externas.

In [16]:
import joblib
from sklearn.pipeline import Pipeline

# 1. Unimos la máquina de limpieza y el modelo en un solo "Súper Tubo" (Pipeline final)
# Así, cuando llegue un dato nuevo, se limpiará y predecirá automáticamente.
pipeline_definitivo = Pipeline([
    ('preprocesamiento', pipeline_maestro),
    ('modelo', modelo_final)
])

# 2. Guardamos este súper tubo en la carpeta 'models/'
joblib.dump(pipeline_definitivo, '../models/modelo_california.joblib')

print("¡Modelo guardado con éxito! Revisa tu carpeta 'models/'")

¡Modelo guardado con éxito! Revisa tu carpeta 'models/'
